In [ ]:
# CSE 144 Final Project
# Transfer Learning Challenge

import os
import csv
import random
from collections import defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms, datasets
from tqdm.auto import tqdm


# =========================================================
# 0. Reproducibility
# =========================================================
SEED = 999

def set_seed(seed: int = 999):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning:", e)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


# =========================================================
# 1. Paths / constants
# =========================================================
DATA_DIR = os.path.join(os.getcwd(), "ucsc-cse-144-winter-2026-final-project")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

NUM_CLASSES = 100
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2 if device == "cuda" else 0
NUM_EPOCHS = 25
FREEZE_EPOCHS = 5
PATIENCE = 7
MODEL_PATH = os.path.join(os.getcwd(), f"best_model_seed{SEED}.pt")

# =========================================================
# 2. Transforms
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================================================
# 3. Custom ImageFolder with fixed numeric class mapping
# =========================================================
class NumericImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = [d.name for d in os.scandir(directory) if d.is_dir()]
        classes = sorted(classes, key=lambda x: int(x))
        class_to_idx = {cls_name: int(cls_name) for cls_name in classes}
        return classes, class_to_idx


# =========================================================
# 4. Dataset wrapper
# =========================================================
class TransformSubset(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        path, label = self.base_dataset.samples[idx]
        image = self.base_dataset.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


# =========================================================
# 5. Class-balanced split: exactly 2 val images/class
# =========================================================
base_dataset = NumericImageFolder(TRAIN_DIR, transform=None)

for i in range(NUM_CLASSES):
    assert base_dataset.class_to_idx[str(i)] == i, f"Class mapping broken for class {i}"

label_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(base_dataset.samples):
    label_to_indices[label].append(idx)

train_indices = []
val_indices = []

rng = random.Random(42)
for label in range(NUM_CLASSES):
    indices = label_to_indices[label]
    rng.shuffle(indices)

    val_count = 2 if len(indices) >= 3 else 1
    val_indices.extend(indices[:val_count])
    train_indices.extend(indices[val_count:])

train_dataset = TransformSubset(base_dataset, train_indices, transform=train_transform)
val_dataset = TransformSubset(base_dataset, val_indices, transform=val_transform)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))


# =========================================================
# 6. DataLoaders
# =========================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda")
)


# =========================================================
# 7. Model: ResNet18
# =========================================================
def get_model(num_classes=NUM_CLASSES, freeze_backbone=True):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(model.fc.in_features, num_classes)
    )

    for param in model.fc.parameters():
        param.requires_grad = True

    return model


def unfreeze_last_block(model):
    for param in model.layer4.parameters():
        param.requires_grad = True


model = get_model().to(device)
print(model)


# =========================================================
# 8. Loss / optimizer / scheduler
# =========================================================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=2e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# =========================================================
# 9. Train / eval functions
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


# =========================================================
# 10. Training loop
# =========================================================
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_val_acc = 0.0
best_epoch = -1
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    if epoch == FREEZE_EPOCHS:
        print("\nUnfreezing ResNet layer4...\n")
        unfreeze_last_block(model)

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=3e-4,
            weight_decay=2e-4
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"lr: {current_lr:.6f} | "
        f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | "
        f"val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_no_improve = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "class_to_idx": base_dataset.class_to_idx
        }, MODEL_PATH)

        print(f"Saved best checkpoint to {MODEL_PATH}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

print(f"\nBest epoch: {best_epoch}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best val acc:  {best_val_acc:.4f}")


# =========================================================
# 11. Plot curves
# =========================================================
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


# =========================================================
# 12. Kaggle test dataset
# =========================================================
class TestImageDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted(
            [f for f in os.listdir(test_dir) if f.endswith(".jpg")],
            key=lambda x: int(os.path.splitext(x)[0])
        )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        fname = self.image_files[idx]
        path = os.path.join(self.test_dir, fname)
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, fname


test_dataset = TestImageDataset(TEST_DIR, transform=val_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda")
)


# =========================================================
# 13. Prediction with simple TTA
# =========================================================
@torch.no_grad()
def predict_test(model_path, device="cpu"):
    model = get_model(num_classes=NUM_CLASSES, freeze_backbone=False).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    predictions = []

    for images, fnames in tqdm(test_loader):
        images = images.to(device)

        flipped_images = torch.flip(images, dims=[3])

        outputs_1 = model(images)
        outputs_2 = model(flipped_images)
        outputs = (outputs_1 + outputs_2) / 2.0

        preds = outputs.argmax(dim=1).cpu().tolist()

        for fname, pred in zip(fnames, preds):
            predictions.append((fname, pred))

    out_path = os.path.join(os.getcwd(), "submission.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Label"])
        writer.writerows(predictions)

    print(f"Saved submission to {out_path}")
    return out_path


submission_path = predict_test(MODEL_PATH, device=device)
print("Submission file:", submission_path)

In [3]:
import os
import csv
import torch
from tqdm import tqdm

MODEL_PATHS = [
    os.path.join(os.getcwd(), "best_model_seed42.pt"),
    # os.path.join(os.getcwd(), "best_model_seed123.pt"),
    os.path.join(os.getcwd(), "best_model_seed999.pt"),
]

@torch.no_grad()
def predict_test_ensemble(model_paths, device="cpu"):
    ensemble_models = []

    for model_path in model_paths:
        model = get_model(num_classes=NUM_CLASSES, freeze_backbone=False).to(device)
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()
        ensemble_models.append(model)

    predictions = []

    for images, fnames in tqdm(test_loader):
        images = images.to(device)
        flipped_images = torch.flip(images, dims=[3])  # horizontal flip TTA

        avg_logits = None

        for model in ensemble_models:
            logits_original = model(images)
            logits_flipped = model(flipped_images)
            logits = (logits_original + logits_flipped) / 2.0

            if avg_logits is None:
                avg_logits = logits
            else:
                avg_logits += logits

        avg_logits = avg_logits / len(ensemble_models)
        preds = avg_logits.argmax(dim=1).cpu().tolist()

        for fname, pred in zip(fnames, preds):
            predictions.append((fname, pred))

    out_path = os.path.join(os.getcwd(), "submission_ensemble.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Label"])
        writer.writerows(predictions)

    print(f"Saved ensemble submission to {out_path}")
    return out_path

submission_path = predict_test_ensemble(MODEL_PATHS, device=device)
print("Submission file:", submission_path)

100%|██████████| 33/33 [01:17<00:00,  2.33s/it]

Saved ensemble submission to /Users/jonathanperez/cse-144/project/cse144-final/submission_ensemble.csv
Submission file: /Users/jonathanperez/cse-144/project/cse144-final/submission_ensemble.csv
